# Data Loading

In [1]:
%%time

import kplex_dev
import data_splits
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split



warnings.filterwarnings('ignore')

timeout_replacement = 1800  # max timeout replacement
excel_path = "kplex_ML_data.xlsx"

feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms


CPU times: total: 3.67 s
Wall time: 4.77 s


In [2]:
# Loading and preprocessing data
cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)
kplex_dev.process_kcore_list_to_statistics(cat_df)

# Selecting weight scheme
epsilon = 0.001 # need to set !!!
scheme = kplex_dev.W_TIME_MIN_X_W_LOG_TIME_RATIO # Change this and analyse!!

# try dense cross entrpy loss for NN
# take 10% of data out as unseen data and run triaing and test split on remaining 90%data

# Assigning weights
kplex_dev.assign_weights(cat_df, scheme, epsilon)

# kplex_dev.export_sorted(cat_df)

# Data reformatting
cat_df = kplex_dev.post_process_data(cat_df)

# fastest_time = minimum across those algorithms
cat_df['fastest_time'] = cat_df[data_splits.algorithms].min(axis=1)
cat_df

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br,fastest_time
0,MANN-a81,20,3321,5506380,1658.0488,3317,3280,3280,1,3321.000000,3321,3321,0.000000,306.046574,1459.5,1800.0,1468.1,1477.4,1504.7,1459.5
1,wikipedia_link_sr,20,1810,1637059,904.4525,1809,1798,1795,4,1806.250000,1810,1803,2.861381,66.902739,22.6,18.0,737.7,740.4,738.1,18.0
2,scc_reality,20,1332,880733,661.2110,1331,1235,1230,6,1290.333333,1332,1236,35.714921,47.684856,1800.0,1214.8,9.4,9.7,9.0,9.0
3,web-wikipedia_link_en13-all,20,432,93085,215.4745,431,427,427,1,432.000000,432,432,0.000000,184.662528,239.3,287.1,517.7,504.4,516.7,239.3
4,livejournal-links,20,442,94435,213.6538,441,371,369,3,412.333333,442,386,22.983086,0.269931,1.3,1.6,1.6,1.6,1.6,1.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,2,264,11792,44.6667,150,67,52,16,215.000000,264,125,56.656862,0.784308,2.0,0.6,0.5,0.5,2.4,0.5
756,edit-ukwiki,2,281,14183,50.4733,280,65,40,26,235.384615,281,168,34.703222,0.437734,1.2,1.1,0.6,0.6,0.5,0.5
757,johnson8-4-4,2,70,1855,26.5000,53,53,53,1,70.000000,70,70,0.000000,0.389785,0.8,0.9,1.1,1.1,0.3,0.3
758,hamming6-2,2,64,1824,28.5000,57,57,57,1,64.000000,64,64,0.000000,0.081093,0.2,0.3,0.2,0.2,0.3,0.2


# Training with LR

LR with soft labels and with a soft‑cross‑entropy loss.


### Model

In [15]:
# kplex_dev.py

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

# ——— 1) Soft Cross‐Entropy with Temperature --------------------------

class SoftCrossEntropyLoss(nn.Module):
    """
    Cross‐entropy loss where logits are divided by temperature T before softmax.
    Supports both hard targets (ints) and soft targets (probabilities).
    """
    def __init__(self, temperature: float = 1.0):
        super().__init__()
        self.temperature = temperature

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        # scale logits
        scaled = logits / self.temperature
        if targets.dtype in (torch.float32, torch.float64):
            # soft‐target case: targets is (batch, C) probability distribution
            log_probs = F.log_softmax(scaled, dim=1)
            # sum over classes, then mean over batch
            return -(targets * log_probs).sum(dim=1).mean()
        else:
            # hard‐target case: targets is (batch,) of ints
            return F.cross_entropy(scaled, targets)


# ——— 2) sklearn‐style Torch Logistic Regression -----------------------

class TorchLogisticClassifier(BaseEstimator, ClassifierMixin):
    """
    A sklearn‐compatible PyTorch logistic regression with L1/L2/Elastic-Net
    and soft‐CE temperature scaling.  Supports hard or soft labels.
    """
    def __init__(
        self,
        penalty: str=None,       # none,'l1', 'l2', or 'elasticnet'
        alpha: float = 1e-3,     # regularization strength
        beta: float = 0.5,       # for elastic-net
        lr: float = 1e-2,
        batch_size: int = 64,
        epochs: int = 50,
        temperature: float = 1.0,
        device: str = None,
        random_state: int = 42
    ):
        self.penalty = penalty
        self.alpha = alpha
        self.beta = beta
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.temperature = temperature
        self.device = device
        self.random_state = random_state

    def fit(self, X, y, soft_labels: bool = False):
        """
        X : array-like (n_samples, n_features)
        y : if soft_labels=False, array-like of ints (n_samples,)
            if soft_labels=True, array-like of floats (n_samples, n_classes), rows sum to 1
        """
        # 1) Basic setup
        X = check_array(X, force_all_finite="allow-nan")
        n_samples, n_features = X.shape

        # 2) Device & seed
        self.device_ = torch.device(self.device or ("cuda" if torch.cuda.is_available() else "cpu"))
        torch.manual_seed(self.random_state)

        # 3) Scale features
        self.scaler_ = StandardScaler().fit(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)

        # 4) Prepare targets
        if soft_labels:
            # assume y is already array-like floats summing to 1 per row
            y_arr = np.asarray(y, dtype=float)
            n_classes = y_arr.shape[1]
            y_tensor = torch.tensor(y_arr, dtype=torch.float32, device=self.device_)
            # we won't need class_to_idx_ in this mode
        else:
            # hard labels
            y_arr = np.asarray(y)
            self.classes_ = np.unique(y_arr)
            n_classes = len(self.classes_)
            self.class_to_idx_ = {c: i for i, c in enumerate(self.classes_)}
            y_idx = np.vectorize(self.class_to_idx_.get)(y_arr)
            y_tensor = torch.tensor(y_idx, dtype=torch.long, device=self.device_)

        # 5) DataLoader
        dataset = TensorDataset(X_tensor, y_tensor)
        loader  = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        # 6) Model, loss, optimizer
        self.model_ = nn.Linear(n_features, n_classes).to(self.device_)
        criterion = SoftCrossEntropyLoss(self.temperature)
        optimizer = optim.SGD(self.model_.parameters(), lr=self.lr)

        # 7) Training loop
        for epoch in range(1, self.epochs + 1):
            self.model_.train()
            for xb, yb in loader:
                optimizer.zero_grad()
                logits = self.model_(xb)
                loss   = criterion(logits, yb)

                # regularization
                l1 = sum(p.abs().sum()   for p in self.model_.parameters())
                l2 = sum(p.pow(2).sum()  for p in self.model_.parameters())
                if self.penalty == "l1":
                    loss = loss + self.alpha * l1
                elif self.penalty == "l2":
                    loss = loss + self.alpha * l2
                elif self.penalty == "elasticnet":
                    loss = loss + self.alpha * (self.beta*l1 + (1-self.beta)*l2)

                loss.backward()
                optimizer.step()

        return self

    def predict(self, X):
        check_is_fitted(self, ["model_", "scaler_"])
        X = check_array(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        self.model_.eval()
        with torch.no_grad():
            logits = self.model_(X_tensor)
            preds  = logits.argmax(dim=1).cpu().numpy()

        # map back to original classes if hard‐label mode
        if hasattr(self, "class_to_idx_"):
            inv = {i:c for c,i in self.class_to_idx_.items()}
            return np.vectorize(inv.get)(preds)
        return preds

    def predict_proba(self, X):
        check_is_fitted(self, ["model_", "scaler_"])
        X = check_array(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        self.model_.eval()
        with torch.no_grad():
            scaled = self.model_(X_tensor) / self.temperature
            return F.softmax(scaled, dim=1).cpu().numpy()


In [4]:
def expand_with_ties_and_prepare(
    df,
    weight_col='weight',
    target_col_name='target',
    for_nn=False,
    temprature=1.0,
    tie_tolerance: float = 0.0   # 0% default
):
    """
    Given a DataFrame with multiple algorithm runtimes,
    expand rows to duplicate ties (within tolerance) and adjust weights,
    then return X, hard y, soft y, and sample weights.

    Returns
    -------
    X_out        : pd.DataFrame
    y_out        : pd.Series         # hard labels (one per expanded row)
    y_soft_out   : np.ndarray        # soft targets (same shape as X_out × n_algos)
    weights_out  : pd.Series
    """
    expanded_rows = []
    hard_labels   = []
    soft_labels   = []
    weights_list  = []

    # precompute feature set & number of algorithms
    n_alg = len(algorithms)

    for _, row in df.iterrows():
        # 1) raw runtimes and threshold
        runtimes = np.array([row[alg] for alg in algorithms], dtype=float)
        min_rt    = runtimes.min()
        thresh    = (1 + tie_tolerance) * min_rt

        # 2) hard‐tie set
        hard_set = [alg for alg, rt in zip(algorithms, runtimes) if rt <= thresh]

        # 3) soft‐target over *all* algos = softmax(1/runtime)
        inv = 1.0 / (runtimes + 1e-8)
        ex  = np.exp(inv/temprature)
        soft = ex / ex.sum()

        # 4) one expanded row per hard‐tie
        for alg in hard_set:
            if 'kcore_list' in row:
                new_row = row[feature_cols + ['kcore_list']].copy()
            elif for_nn:
                new_row = row[feature_cols + ['fastest_time']].copy()
            else:
                new_row = row[feature_cols].copy()

            expanded_rows.append(new_row)
            hard_labels.append(alg)
            soft_labels.append(soft)   # same soft vector for each copy
            weights_list.append(row[weight_col] / len(hard_set))

    expanded_df = pd.DataFrame(expanded_rows)

    X_out       = expanded_df.drop([target_col_name, weight_col], axis=1, errors='ignore')
    y_out       = pd.Series(hard_labels,  name=target_col_name)
    y_soft_out  = np.vstack(soft_labels)          # shape (n_expanded, n_alg)
    weights_out = pd.Series(weights_list, name=weight_col)

    return X_out, y_out, y_soft_out, weights_out


In [21]:
import pandas as pd
import numpy as np

def torch_logistic_feature_importances(clf, feature_names):
    """
    Given a fitted TorchLogisticClassifier `clf` and the list of feature names,
    returns a DataFrame of absolute average weights per feature, sorted descending.
    """
    # Grab the learned weight tensor (shape: n_classes x n_features)
    W = clf.model_.weight.detach().cpu().numpy()   # ndarray of shape (C, F)
    
    # Compute per-Coefficients (Linear Models): e.g. mean of absolute weights across classes
    coefficents = np.mean(np.abs(W), axis=0)       # shape (F,)
    
    # Build DataFrame
    coef_df = pd.DataFrame({
        'feature': feature_names,
        'coefficents': coefficents
    }).sort_values('coefficents', ascending=False).reset_index(drop=True)
    
    return coef_df



## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)

**with_holdout = True**



### **Balanced train-test split based on Y**

In [5]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 21)
Test set shape:  (156, 21)


In [10]:
from sklearn.preprocessing import LabelEncoder

X_train_, y_train_hard, y_train_soft, weights_out = expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.0,
    temprature=1
)

from sklearn.preprocessing import LabelEncoder
# Label‐encode the string targets into integers 0…4 ---
le = LabelEncoder()
y_train_hard = le.fit_transform(y_train_hard)     # e.g. ['AltRB','UMKP',…] → [0,3,…]


print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


#### LR without regularization


In [ ]:

clf_LR_no_penalty_strat_y = TorchLogisticClassifier(
        penalty=None,
        lr=0.01,
        batch_size=32,
        epochs=1000,
    )

clf_LR_no_penalty_strat_y.fit(X_train_, y_train_soft, soft_labels=True)
# clf_LR_no_penalty_strat_y.fit(X_train_, y_train_


TorchLogisticClassifier(batch_size=32, epochs=1000)

Coefficients (Linear Models)

In [22]:
feature_cols = X_train_.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_LR_no_penalty_strat_y, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.960803
1   shell_stdSize     0.551376
2   shell_minSize     0.531163
3   shell_avgSize     0.485626
4               n     0.407957
5       shell_num     0.371430
6           d_avg     0.363943
7        core_min     0.316248
8   shell_maxSize     0.189973
9        core_max     0.167752
10          d_max     0.147402
11        k_value     0.127527


#### Inference on 20% dataset

In [23]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_LR_no_penalty_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/No_penalty/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/No_penalty/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 56.41%

Precision by Algorithm:
  UMKP: 75.00% (4 predictions)
  AltRB: 68.75% (16 predictions)
  kPEX: 56.76% (111 predictions)
  Pivot_Br: 0.00% (0 predictions)
  S_Br: 44.00% (25 predictions)

Recall by Algorithm:
  UMKP: 50.00% (10/20 cases)
  AltRB: 60.87% (14/23 cases)
  kPEX: 91.78% (67/73 cases)
  Pivot_Br: 46.67% (21/45 cases)
  S_Br: 49.02% (25/51 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 23203.70
  Total Optimal Time   = 16592.90
  Time Ratio (pred/opt)= 1.398
  Avg Overhead/case ((pred - opt) / opt)    = 4365.53%


Adding prediction probabilities...

Saving full predictions to './LR_Results_soft_labels_soft_labels/tie_tol/0/No_penalty/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 68 wrong predictions with

#### Re-evalute on entire dataset

In [24]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_LR_no_penalty_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/No_penalty/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/No_penalty/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 56.05%

Precision by Algorithm:
  UMKP: 51.72% (29 predictions)
  AltRB: 56.99% (93 predictions)
  kPEX: 57.44% (531 predictions)
  Pivot_Br: 0.00% (0 predictions)
  S_Br: 49.53% (107 predictions)

Recall by Algorithm:
  UMKP: 46.00% (46/100 cases)
  AltRB: 63.79% (74/116 cases)
  kPEX: 90.36% (328/363 cases)
  Pivot_Br: 53.54% (121/226 cases)
  S_Br: 48.05% (123/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 99905.30
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.494
  Avg Overhead/case ((pred - opt) / opt)    = 246.63%


Adding prediction probabilities...

Saving full predictions to './LR_Results_soft_labels_soft_labels/tie_tol/0/No_penalty/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 334 wrong pre

#### LR with L1 regularization

In [ ]:
# Instantiate and train
clf_l1_strat_y = TorchLogisticClassifier(
        penalty='l1',
        alpha=1e-3,
        lr=1e-2,
        batch_size=128,
        epochs=1000,
    )
# clf_l1_strat_y.fit(X_train, y_train)
clf_l1_strat_y.fit(X_train_, y_train_soft, soft_labels=True)



TorchLogisticClassifier(batch_size=128, epochs=1000, penalty='l1')

Coefficients (Linear Models)

In [ ]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_l1_strat_y, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.391230
1       shell_num     0.206233
2           d_max     0.115297
3        core_min     0.110313
4   shell_stdSize     0.095252
5   shell_maxSize     0.075224
6           d_avg     0.065593
7         k_value     0.056383
8   shell_minSize     0.048584
9   shell_avgSize     0.025902
10       core_max     0.019912
11              n     0.015847


#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_l1_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/L1/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/L1/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.71%

Precision by Algorithm:
  UMKP: 50.00% (12 predictions)
  AltRB: 76.47% (17 predictions)
  kPEX: 58.33% (300 predictions)
  Pivot_Br: 16.88% (77 predictions)
  S_Br: 42.86% (14 predictions)

Recall by Algorithm:
  UMKP: 47.17% (25/53 cases)
  AltRB: 39.13% (27/69 cases)
  kPEX: 80.72% (180/223 cases)
  Pivot_Br: 69.68% (108/155 cases)
  S_Br: 49.32% (72/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 56369.30
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.215
  Avg Overhead/case ((pred - opt) / opt)    = 137.47%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 207 wrong predictions with time metrics

Top 5 he

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_l1_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/L1/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/L1/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 46.84%

Precision by Algorithm:
  UMKP: 59.38% (32 predictions)
  AltRB: 63.33% (30 predictions)
  kPEX: 53.96% (530 predictions)
  Pivot_Br: 18.62% (145 predictions)
  S_Br: 21.74% (23 predictions)

Recall by Algorithm:
  UMKP: 37.00% (37/100 cases)
  AltRB: 31.90% (37/116 cases)
  kPEX: 81.27% (295/363 cases)
  Pivot_Br: 64.16% (145/226 cases)
  S_Br: 39.45% (101/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 102641.80
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.535
  Avg Overhead/case ((pred - opt) / opt)    = 234.85%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 404 wrong predictions with time metrics



#### LR with L2 regularization

In [ ]:
# Instantiate and train
clf_l2_strat_y = TorchLogisticClassifier(
        penalty='l2',
        alpha=1e-2,
        beta=0.7,
        lr=1e-2,
        batch_size=128,
        epochs=100,
    )
clf_l2_strat_y.fit(X_train_, y_train_soft, soft_labels=True)


TorchLogisticClassifier(alpha=0.01, batch_size=128, beta=0.7, epochs=100,
                        penalty='l2')

Coefficients (Linear Models)

In [ ]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_l2_strat_y, feature_cols)
print(coef_df)

          feature  coefficents
0        core_min     0.136232
1   shell_avgSize     0.131512
2       shell_num     0.127192
3   shell_stdSize     0.122392
4               m     0.118859
5               n     0.107214
6           d_avg     0.106994
7         k_value     0.101776
8   shell_minSize     0.095497
9   shell_maxSize     0.079584
10       core_max     0.065749
11          d_max     0.048722


#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_l2_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/L2/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 52.62%

Precision by Algorithm:
  UMKP: 0.00% (2 predictions)
  AltRB: 44.12% (34 predictions)
  kPEX: 60.47% (301 predictions)
  Pivot_Br: 22.95% (61 predictions)
  S_Br: 45.45% (22 predictions)

Recall by Algorithm:
  UMKP: 37.74% (20/53 cases)
  AltRB: 46.38% (32/69 cases)
  kPEX: 85.20% (190/223 cases)
  Pivot_Br: 67.74% (105/155 cases)
  S_Br: 49.32% (72/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 61089.60
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.316
  Avg Overhead/case ((pred - opt) / opt)    = 214.44%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L2/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 199 wrong predictions with time metrics

Top 5 heav

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_l2_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/L2/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 48.55%

Precision by Algorithm:
  UMKP: 63.64% (11 predictions)
  AltRB: 46.15% (65 predictions)
  kPEX: 54.91% (550 predictions)
  Pivot_Br: 21.50% (107 predictions)
  S_Br: 25.93% (27 predictions)

Recall by Algorithm:
  UMKP: 29.00% (29/100 cases)
  AltRB: 41.38% (48/116 cases)
  kPEX: 86.23% (313/363 cases)
  Pivot_Br: 61.95% (140/226 cases)
  S_Br: 39.06% (100/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 104756.30
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.566
  Avg Overhead/case ((pred - opt) / opt)    = 258.42%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L2/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 391 wrong predictions with time metrics



#### LR with L1+L2 regularization

$loss + \alpha \times (\beta \times l_1 + (1 - \beta) \times l_2)$

In [ ]:
# Instantiate and train
clf_elasticnet_strat_y = TorchLogisticClassifier(
        penalty='elasticnet',
        alpha=1e-3,
        beta=0.7,
        lr=1e-2,
        batch_size=128, # batch size = X.shape
        epochs=100,
    )
clf_elasticnet_strat_y.fit(X_train_, y_train_soft, soft_labels=True)


TorchLogisticClassifier(batch_size=128, beta=0.7, epochs=100,
                        penalty='elasticnet')

Coefficients (Linear Models)

In [ ]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_elasticnet_strat_y, feature_cols)
print(coef_df)

          feature  coefficents
0        core_min     0.162781
1   shell_avgSize     0.162600
2       shell_num     0.136016
3   shell_stdSize     0.132233
4               n     0.128314
5               m     0.126527
6           d_avg     0.125410
7   shell_minSize     0.118283
8         k_value     0.115459
9   shell_maxSize     0.099194
10       core_max     0.076786
11          d_max     0.051268


#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_elasticnet_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/L1+L2/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/L1+L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 52.38%

Precision by Algorithm:
  UMKP: 0.00% (2 predictions)
  AltRB: 44.12% (34 predictions)
  kPEX: 60.87% (299 predictions)
  Pivot_Br: 20.97% (62 predictions)
  S_Br: 43.48% (23 predictions)

Recall by Algorithm:
  UMKP: 37.74% (20/53 cases)
  AltRB: 46.38% (32/69 cases)
  kPEX: 85.20% (190/223 cases)
  Pivot_Br: 67.10% (104/155 cases)
  S_Br: 49.32% (72/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 61020.70
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.315
  Avg Overhead/case ((pred - opt) / opt)    = 212.18%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1+L2/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 200 wrong predictions with time metrics

Top 5 h

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_elasticnet_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/L1+L2/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/L1+L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 48.29%

Precision by Algorithm:
  UMKP: 63.64% (11 predictions)
  AltRB: 45.31% (64 predictions)
  kPEX: 55.29% (548 predictions)
  Pivot_Br: 19.63% (107 predictions)
  S_Br: 23.33% (30 predictions)

Recall by Algorithm:
  UMKP: 29.00% (29/100 cases)
  AltRB: 40.52% (47/116 cases)
  kPEX: 86.50% (314/363 cases)
  Pivot_Br: 61.06% (138/226 cases)
  S_Br: 39.06% (100/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 106157.90
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.587
  Avg Overhead/case ((pred - opt) / opt)    = 301.73%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1+L2/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 393 wrong predictions with time metric

In [ ]:
# ## For exporting classification reports to a temp XL file.

# from write_utils import append_report_to_excel

# append_report_to_excel(
#     filename="temp_results.xlsx",
#     heading="Balanced train-test split based on Y ",
#     classif_report_df=classif_df,
#     feat_importances_df=importances,
#     sheet_name="Sheet1"    # or whatever your sheet is called
# )


### **Balanced train-test split based on both Y and k**

In [ ]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [ ]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train, y_train, weights_train = expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train = le.fit_transform(y_train)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


#### LR without regularization

In [ ]:
# Instantiate and train
clf_strat_y_K = TorchLogisticClassifier(
    penalty=None, # None, 'l1','l2', or 'elasticnet'
    alpha=0,           # regularization strength
    lr=0.25,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time. 
    batch_size=128, 
    epochs=1000,
)
clf_strat_y_K.fit(X_train_, y_train_soft, soft_labels=True)

TorchLogisticClassifier(alpha=0, batch_size=128, epochs=1000, lr=0.25)

Coefficients (Linear Models)

In [ ]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_strat_y_K, feature_cols)
print(coef_df)

          feature  coefficents
0           d_avg     1.706208
1        core_min     0.984353
2        core_max     0.818618
3   shell_avgSize     0.731164
4   shell_maxSize     0.593695
5               m     0.589565
6               n     0.375709
7           d_max     0.230942
8       shell_num     0.190099
9   shell_minSize     0.179037
10        k_value     0.133226
11  shell_stdSize     0.072736


#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/No_penalty/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/No_penalty/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 52.62%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 67.86% (28 predictions)
  kPEX: 58.90% (292 predictions)
  Pivot_Br: 20.00% (35 predictions)
  S_Br: 35.38% (65 predictions)

Recall by Algorithm:
  UMKP: 33.96% (18/53 cases)
  AltRB: 47.83% (33/69 cases)
  kPEX: 85.20% (190/223 cases)
  Pivot_Br: 62.58% (97/155 cases)
  S_Br: 51.37% (75/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 55020.10
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.186
  Avg Overhead/case ((pred - opt) / opt)    = 134.50%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/No_penalty/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 199 wrong predictions with time metrics

T

Metrics written to ./LR_results/tie_tol/50/No_penalty/split_and_expand/model_performance_for_all_experiments.xlsx


#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/No_penalty/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/No_penalty/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.92%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 56.45% (62 predictions)
  kPEX: 55.29% (539 predictions)
  Pivot_Br: 23.88% (67 predictions)
  S_Br: 41.30% (92 predictions)

Recall by Algorithm:
  UMKP: 23.00% (23/100 cases)
  AltRB: 44.83% (52/116 cases)
  kPEX: 88.98% (323/363 cases)
  Pivot_Br: 58.41% (132/226 cases)
  S_Br: 46.48% (119/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 97558.70
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.459
  Avg Overhead/case ((pred - opt) / opt)    = 234.38%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/No_penalty/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 373 wrong predictions with time met

#### LR with L1 regularization

In [ ]:
# Instantiate and train
clf_l1_strat_y_K = TorchLogisticClassifier(
    penalty='l1', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.25,              # Learning rate
    batch_size=128,        
    epochs=1000,
)
clf_l1_strat_y_K.fit(X_train_, y_train_soft, soft_labels=True)


TorchLogisticClassifier(batch_size=128, epochs=1000, lr=0.25, penalty='l1')

Coefficients (Linear Models)

In [ ]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_l1_strat_y_K, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.340855
1       shell_num     0.193455
2           d_max     0.163610
3   shell_minSize     0.132404
4        core_min     0.102922
5         k_value     0.063169
6           d_avg     0.043792
7   shell_stdSize     0.039189
8               n     0.022302
9   shell_maxSize     0.022287
10       core_max     0.018797
11  shell_avgSize     0.013350


#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_l1_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/L1/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/L1/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 53.81%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 55.56% (27 predictions)
  kPEX: 60.40% (303 predictions)
  Pivot_Br: 25.58% (43 predictions)
  S_Br: 36.17% (47 predictions)

Recall by Algorithm:
  UMKP: 32.08% (17/53 cases)
  AltRB: 46.38% (32/69 cases)
  kPEX: 87.89% (196/223 cases)
  Pivot_Br: 63.23% (98/155 cases)
  S_Br: 50.68% (74/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 57454.10
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.238
  Avg Overhead/case ((pred - opt) / opt)    = 199.15%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 194 wrong predictions with time metrics

Top 5 hea

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_l1_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/L1/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/L1/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.66%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 51.72% (58 predictions)
  kPEX: 56.06% (553 predictions)
  Pivot_Br: 23.17% (82 predictions)
  S_Br: 38.81% (67 predictions)

Recall by Algorithm:
  UMKP: 22.00% (22/100 cases)
  AltRB: 41.38% (48/116 cases)
  kPEX: 90.91% (330/363 cases)
  Pivot_Br: 59.29% (134/226 cases)
  S_Br: 43.75% (112/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 102499.30
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.533
  Avg Overhead/case ((pred - opt) / opt)    = 296.78%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...



Saved 375 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
          dataset  k_value      weight predicted fastest_algorithms  \
646    bag-pubmed        3  662.182989  Pivot_Br             [UMKP]   
491       keller4        5  660.562496      kPEX             [S_Br]   
470   edit-itwiki        5  660.087006      kPEX             [S_Br]   
570       keller4        4  657.474286      kPEX             [S_Br]   
684  wiki_talk_en        2  651.742897      kPEX         [Pivot_Br]   

     time_ratio  
646    2.718623  
491    2.679591  
470    1.255654  
570    3.077975  
684    1.066777  

Error Analysis:
Average time ratio: 24.87x slower
Worst time ratio: 6700.00x slower
Best time ratio: 1.00x slower
Median time gap: 2.10 seconds
Largest time gap: 1797.10 seconds
Smallest time gap: 0.10 seconds

Error Severity Distribution:
time_ratio
≤5%         80
5–20%      114
20–100%     86
>100%       95
Name: count, dtype: int64

Algorithm‐wise Errors:
  UMKP: 0 errors, avg

#### LR with L2 regularization

In [ ]:
# Instantiate and train
clf_l2_strat_y_K = TorchLogisticClassifier(
    penalty='l2', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.25,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time.
    batch_size=128,       # Feeding whole data at a time.
    epochs=1000,
)
clf_l2_strat_y_K.fit(X_train_, y_train_soft, soft_labels=True)


TorchLogisticClassifier(batch_size=128, epochs=1000, lr=0.25, penalty='l2')

Coefficients (Linear Models)

In [ ]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_l2_strat_y_K, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.426148
1           d_avg     0.309265
2       shell_num     0.206892
3        core_min     0.205416
4           d_max     0.203007
5        core_max     0.197777
6   shell_maxSize     0.108241
7               n     0.108043
8   shell_minSize     0.096344
9         k_value     0.091241
10  shell_stdSize     0.062822
11  shell_avgSize     0.060956


#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_l2_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/L2/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 53.33%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 56.00% (25 predictions)
  kPEX: 60.26% (302 predictions)
  Pivot_Br: 25.00% (44 predictions)
  S_Br: 34.69% (49 predictions)

Recall by Algorithm:
  UMKP: 32.08% (17/53 cases)
  AltRB: 44.93% (31/69 cases)
  kPEX: 87.44% (195/223 cases)
  Pivot_Br: 62.58% (97/155 cases)
  S_Br: 50.68% (74/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 55062.70
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.187
  Avg Overhead/case ((pred - opt) / opt)    = 134.86%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L2/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 196 wrong predictions with time metrics

Top 5 hea

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_l2_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/L2/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.66%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 50.82% (61 predictions)
  kPEX: 56.31% (547 predictions)
  Pivot_Br: 23.75% (80 predictions)
  S_Br: 37.50% (72 predictions)

Recall by Algorithm:
  UMKP: 22.00% (22/100 cases)
  AltRB: 42.24% (49/116 cases)
  kPEX: 90.63% (329/363 cases)
  Pivot_Br: 58.85% (133/226 cases)
  S_Br: 43.75% (112/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 101316.80
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.515
  Avg Overhead/case ((pred - opt) / opt)    = 278.40%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L2/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 375 wrong predictions with time metrics

T

#### LR with L1+L2 regularization

$loss + \alpha \times (\beta \times l_1 + (1 - \beta) \times l_2)$

In [ ]:
# Instantiate and train
clf_elasticnet_strat_y_K = TorchLogisticClassifier(
    penalty='elasticnet', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    beta=0.6,         # regularization strength of L1 in usint elasticnet
    lr=0.5,              # Learning rate
    batch_size=128,
    epochs=1000,
)
clf_elasticnet_strat_y_K.fit(X_train_, y_train_soft, soft_labels=True)


TorchLogisticClassifier(batch_size=128, beta=0.6, epochs=1000, lr=0.5,
                        penalty='elasticnet')

Coefficients (Linear Models)

In [ ]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_elasticnet_strat_y_K, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.359199
1       shell_num     0.198811
2           d_max     0.178036
3   shell_minSize     0.100727
4           d_avg     0.100040
5        core_min     0.080962
6         k_value     0.075069
7   shell_stdSize     0.064561
8        core_max     0.063770
9               n     0.053538
10  shell_maxSize     0.053468
11  shell_avgSize     0.025339


#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_elasticnet_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/L1+L2/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/L1+L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 51.19%

Precision by Algorithm:
  UMKP: 16.67% (12 predictions)
  AltRB: 44.44% (9 predictions)
  kPEX: 58.02% (293 predictions)
  Pivot_Br: 33.33% (54 predictions)
  S_Br: 40.38% (52 predictions)

Recall by Algorithm:
  UMKP: 35.85% (19/53 cases)
  AltRB: 30.43% (21/69 cases)
  kPEX: 85.20% (190/223 cases)
  Pivot_Br: 65.81% (102/155 cases)
  S_Br: 49.32% (72/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 54890.70
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.183
  Avg Overhead/case ((pred - opt) / opt)    = 113.38%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1+L2/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 205 wrong predictions with time metrics

Top 

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_elasticnet_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_Results_soft_labels_soft_labels/tie_tol/0/L1+L2/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_Results_soft_labels_soft_labels/tie_tol/0/L1+L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.53%

Precision by Algorithm:
  UMKP: 40.00% (15 predictions)
  AltRB: 56.25% (32 predictions)
  kPEX: 55.43% (534 predictions)
  Pivot_Br: 27.00% (100 predictions)
  S_Br: 46.84% (79 predictions)

Recall by Algorithm:
  UMKP: 28.00% (28/100 cases)
  AltRB: 31.90% (37/116 cases)
  kPEX: 89.26% (324/363 cases)
  Pivot_Br: 60.62% (137/226 cases)
  S_Br: 45.70% (117/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 99995.20
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.495
  Avg Overhead/case ((pred - opt) / opt)    = 267.07%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1+L2/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 376 wrong predictions with time metri

Metrics written to ./LR_results/tie_tol/50/L1+L2/split_and_expand/model_performance_for_all_experiments.xlsx
